# Final URA-Focused HGB and RF Hyperparameter Tuning

This notebook tunes the final candidate models for the primary target:

- `target_next_URA`

Models tuned:

- `HistGradientBoostingRegressor`
- `RandomForestRegressor`

The selected URA-tuned configurations are evaluated on the held-out test set for URA. Because IRI is a secondary target here, the same selected model configurations can also be refit/evaluated on IRI for reference without doing a separate IRI hyperparameter search.


## 1. Configuration

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional
import inspect
import json
import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

MODEL_DATA_PATH = Path("data/road_model_dataset_v2.parquet")
EVENT_HISTORY_PATH = Path("data/road_event_history_v2.parquet")
RESULT_DIR = Path("results/final_model_tuning")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

# HGB is memory-efficient enough to tune on full data.
HGB_TRAIN_LIMIT = None

# RF is much more memory-heavy. Tune on a representative sample, then optionally
# train only the selected RF once on full train+validation if the machine can handle it.
RF_TUNING_TRAIN_LIMIT = 600_000
RF_FINAL_TRAIN_LIMIT = 600_000
ALLOW_FULL_RF_FINAL_TRAINING = False

TUNING_TARGET = "target_next_URA"
REFERENCE_TARGETS = ["target_next_IRI"]
EVALUATE_REFERENCE_TARGETS = True
FEATURE_MIXTURE_NAME = "current_static_lag_lifecycle_material"
HORIZON_WINDOWS = {"1y": (274, 457), "2y": (639, 822), "3y": (1004, 1187), "4y": (1370, 1553)}

print(f"Tuning outputs will be written to: {RESULT_DIR.resolve()}")

: 

## 2. Load Data and Add Current Material State

In [ ]:
MODEL_COLUMNS = [
    "Segment_ID", "Lifecycle_ID", "event_date", "ELY",
    "KVL", "KVL_raskas", "KVL_kaista", "Nopeus", "Toim_lk",
    "IRI", "URA", "prev_IRI", "prev_URA", "Delta_t_years",
    "Pavement_Age_years", "Minor_TP_Count", "tp_count_interval",
    "target_next_URA", "target_next_IRI", "target_horizon_days", "target_horizon_years",
]

df = pd.read_parquet(MODEL_DATA_PATH, columns=MODEL_COLUMNS)
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df["observed_lifecycle_age_years"] = pd.to_numeric(df["Pavement_Age_years"], errors="coerce")

SURFACE_TYPE_MAP = {"ab": "AB", "abk": "AB", "abs": "AB", "ea": "AB", "pab-v": "PAB", "pab-b": "PAB", "pab-o": "PAB", "sma": "SMA"}
MATERIAL_TYPE_MAP = {"lta": "new", "mp": "new", "mpkj": "new", "art": "new", "mpk": "new", "rem": "recycled", "urem": "recycled", "remo": "recycled", "rem+": "recycled"}

def normalize_token(value):
    if pd.isna(value):
        return None
    token = str(value).strip().lower()
    return token if token else None

def classify_surface(value):
    token = normalize_token(value)
    return "unknown" if token is None else SURFACE_TYPE_MAP.get(token, "other")

def classify_material(value):
    token = normalize_token(value)
    return "unknown" if token is None else MATERIAL_TYPE_MAP.get(token, "other")

def add_current_material_state(model_df, event_history_path):
    cols = ["event_type", "Segment_ID", "event_date", "Tp_pinta", "Tp_tyomen", "tp_idx"]
    try:
        tp = pd.read_parquet(event_history_path, columns=cols, filters=[("event_type", "==", "TP")])
    except (TypeError, ValueError):
        events = pd.read_parquet(event_history_path, columns=cols)
        tp = events.loc[events["event_type"].eq("TP")].copy()
    tp = tp.dropna(subset=["Segment_ID", "event_date"]).copy()
    tp["event_date"] = pd.to_datetime(tp["event_date"], errors="coerce")
    tp = tp.dropna(subset=["event_date"])
    tp["surface_type_current"] = tp["Tp_pinta"].map(classify_surface)
    tp["material_type_current"] = tp["Tp_tyomen"].map(classify_material)
    tp["tp_idx"] = pd.to_numeric(tp["tp_idx"], errors="coerce")

    left = model_df.reset_index(names="_row_id").sort_values(["event_date", "Segment_ID"], kind="mergesort")
    right = (
        tp[["Segment_ID", "event_date", "tp_idx", "surface_type_current", "material_type_current"]]
        .sort_values(["event_date", "Segment_ID", "tp_idx"], kind="mergesort")
        .rename(columns={"event_date": "latest_material_event_date"})
    )
    merged = pd.merge_asof(
        left, right,
        left_on="event_date", right_on="latest_material_event_date",
        by="Segment_ID", direction="backward", allow_exact_matches=True,
    )
    merged["surface_type_current"] = merged["surface_type_current"].fillna("unknown")
    merged["material_type_current"] = merged["material_type_current"].fillna("unknown")
    merged["years_since_material_update"] = (merged["event_date"] - merged["latest_material_event_date"]).dt.days / 365.25
    return merged.sort_values("_row_id", kind="mergesort").drop(columns=["_row_id", "tp_idx"]).reset_index(drop=True)

df = add_current_material_state(df, EVENT_HISTORY_PATH)
print(f"Rows: {len(df):,}; segments: {df['Segment_ID'].nunique():,}")
display(df.head())

## 3. Split, Features, and Preprocessing

In [ ]:
def make_ely_segment_split(frame, train_frac, val_frac, test_frac, random_state):
    rng = np.random.default_rng(random_state)
    segment_split = {}
    segment_ely = frame[["Segment_ID", "ELY"]].drop_duplicates()
    if segment_ely["Segment_ID"].duplicated().any():
        raise ValueError("Segment_ID appears in multiple ELY groups.")
    for ely, group in segment_ely.groupby("ELY", sort=True):
        segments = group["Segment_ID"].to_numpy(copy=True)
        rng.shuffle(segments)
        n = len(segments)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        n_train = min(max(n_train, 1), n - 2)
        n_val = min(max(n_val, 1), n - n_train - 1)
        for seg in segments[:n_train]:
            segment_split[seg] = "train"
        for seg in segments[n_train:n_train + n_val]:
            segment_split[seg] = "validation"
        for seg in segments[n_train + n_val:]:
            segment_split[seg] = "test"
    return frame["Segment_ID"].map(segment_split).astype("category")

df["split"] = make_ely_segment_split(df, TRAIN_FRACTION, VALIDATION_FRACTION, TEST_FRACTION, RANDOM_STATE)
assert df[["Segment_ID", "split"]].drop_duplicates().groupby("Segment_ID")["split"].nunique().max() == 1

FEATURES = [
    "URA", "IRI", "target_horizon_years",
    "KVL", "KVL_raskas", "KVL_kaista", "Nopeus", "Toim_lk",
    "prev_URA", "prev_IRI", "Delta_t_years",
    "observed_lifecycle_age_years", "Minor_TP_Count",
    "tp_count_interval", "surface_type_current", "material_type_current", "years_since_material_update",
]
CATEGORICAL_FEATURES = {"Toim_lk", "surface_type_current", "material_type_current"}
NUMERIC_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]

def make_one_hot_encoder():
    params = {"handle_unknown": "ignore"}
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        params["sparse_output"] = False
    else:
        params["sparse"] = False
    return OneHotEncoder(**params)

def build_preprocessor():
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", make_one_hot_encoder()),
        ]), [c for c in FEATURES if c in CATEGORICAL_FEATURES]),
    ], remainder="drop", verbose_feature_names_out=True)

train_frame = df.loc[df["split"].eq("train")].copy()
validation_frame = df.loc[df["split"].eq("validation")].copy()
test_frame = df.loc[df["split"].eq("test")].copy()
train_val_frame = df.loc[df["split"].isin(["train", "validation"])].copy()

display(df["split"].value_counts().to_frame("rows"))
display((pd.crosstab(df["split"], df["ELY"], normalize="index") * 100).round(2))

## 4. Candidate Hyperparameter Grids

In [ ]:
HGB_PARAM_GRID = [
    {"learning_rate": 0.03, "max_iter": 650, "max_leaf_nodes": 31, "min_samples_leaf": 50, "l2_regularization": 0.0},
    {"learning_rate": 0.05, "max_iter": 500, "max_leaf_nodes": 31, "min_samples_leaf": 50, "l2_regularization": 0.0},
    {"learning_rate": 0.05, "max_iter": 500, "max_leaf_nodes": 63, "min_samples_leaf": 50, "l2_regularization": 0.0},
    {"learning_rate": 0.05, "max_iter": 500, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.01},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 31, "min_samples_leaf": 50, "l2_regularization": 0.01},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.05},
]

# RF candidates are deliberately memory-safer than the original full-data grid.
# Tuning uses RF_TUNING_TRAIN_LIMIT rows. The chosen candidate can later be trained once on full data.
RF_PARAM_GRID = [
    {"n_estimators": 250, "max_features": 0.5, "min_samples_leaf": 5, "max_depth": None},
    {"n_estimators": 300, "max_features": 0.7, "min_samples_leaf": 8, "max_depth": None},
    {"n_estimators": 350, "max_features": "sqrt", "min_samples_leaf": 5, "max_depth": None},
]

print(f"HGB candidates: {len(HGB_PARAM_GRID)}")
print(f"RF candidates: {len(RF_PARAM_GRID)}")
print(f"RF tuning train limit: {RF_TUNING_TRAIN_LIMIT:,} rows")
print(f"Full RF final training enabled: {ALLOW_FULL_RF_FINAL_TRAINING}")

## 5. Tuning Helpers

In [ ]:
def metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
    }

def sample_frame(frame, limit):
    if limit is None or len(frame) <= limit:
        return frame
    return frame.sample(limit, random_state=RANDOM_STATE)

def make_hgb(params):
    return HistGradientBoostingRegressor(
        loss="squared_error",
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=25,
        random_state=RANDOM_STATE,
        **params,
    )

def make_rf(params):
    return RandomForestRegressor(
        n_jobs=-1,
        random_state=RANDOM_STATE,
        bootstrap=True,
        **params,
    )

def fit_eval(estimator, target, train_data, eval_data, train_limit=None):
    fit_data = sample_frame(train_data, train_limit)
    pipe = Pipeline([("preprocess", build_preprocessor()), ("model", estimator)])
    t0 = time.time()
    pipe.fit(fit_data[FEATURES], fit_data[target])
    fit_seconds = time.time() - t0
    pred = pipe.predict(eval_data[FEATURES])
    return pipe, {
        **metrics(eval_data[target], pred),
        "train_rows": len(fit_data),
        "evaluation_rows": len(eval_data),
        "fit_seconds": fit_seconds,
    }, pred

def tune_grid(model_name, param_grid, target, train_data, eval_data, train_limit):
    rows = []
    for i, params in enumerate(param_grid, start=1):
        print(f"{model_name} {target}: candidate {i}/{len(param_grid)} {params}")
        estimator = make_hgb(params) if model_name == "hist_gradient_boosting" else make_rf(params)
        _, result, _ = fit_eval(estimator, target, train_data, eval_data, train_limit)
        row = {
            "model": model_name,
            "target": target,
            "candidate": i,
            "params": json.dumps(params),
            **result,
        }
        rows.append(row)
        display(pd.DataFrame([row]))
    return pd.DataFrame(rows)

## 6. Tune HistGradientBoostingRegressor

In [ ]:
hgb_results_df = tune_grid(
    "hist_gradient_boosting",
    HGB_PARAM_GRID,
    TUNING_TARGET,
    train_frame,
    validation_frame,
    HGB_TRAIN_LIMIT,
).sort_values(["target", "rmse"])

display(hgb_results_df)
hgb_results_df.to_csv(RESULT_DIR / "hgb_ura_validation_tuning.csv", index=False)

## 7. Tune RandomForestRegressor on a Representative Sample

Random forest tuning is intentionally sampled to avoid memory errors. The best RF candidate can be trained once on full train+validation later by setting `ALLOW_FULL_RF_FINAL_TRAINING = True`, but the default is safe and reproducible.

In [ ]:
rf_results_df = tune_grid(
    "random_forest",
    RF_PARAM_GRID,
    TUNING_TARGET,
    train_frame,
    validation_frame,
    RF_TUNING_TRAIN_LIMIT,
).sort_values(["target", "rmse"])

display(rf_results_df)
rf_results_df.to_csv(RESULT_DIR / "rf_ura_validation_tuning_sampled.csv", index=False)

## 8. Select URA-Tuned Final Candidates

In [ ]:
tuning_results = pd.concat([hgb_results_df, rf_results_df], ignore_index=True)
best_validation = (
    tuning_results.sort_values("rmse")
    .groupby(["model"], as_index=False)
    .first()
    .sort_values("rmse")
)
display(best_validation)
tuning_results.to_csv(RESULT_DIR / "all_ura_validation_tuning_results.csv", index=False)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=tuning_results, x="candidate", y="rmse", hue="model")
plt.title("URA hyperparameter tuning validation RMSE")
plt.xlabel("Candidate")
plt.ylabel("Validation RMSE for target_next_URA")
plt.tight_layout()
plt.show()

## 9. Optional Delta Target Check for Tuned HGB on URA

In [14]:
delta_rows = []
target = TUNING_TARGET
current_col = "URA"
best_hgb = best_validation[best_validation["model"].eq("hist_gradient_boosting")].iloc[0]
params = json.loads(best_hgb["params"])

train_delta = train_frame.copy()
validation_delta = validation_frame.copy()
delta_col = f"delta_{target}"
train_delta[delta_col] = train_delta[target] - train_delta[current_col]
validation_delta[delta_col] = validation_delta[target] - validation_delta[current_col]

pipe, result, pred_delta = fit_eval(make_hgb(params), delta_col, train_delta, validation_delta, HGB_TRAIN_LIMIT)
pred_actual = validation_delta[current_col].to_numpy() + pred_delta
actual_metrics = metrics(validation_delta[target], pred_actual)
delta_rows.append({
    "model": "hist_gradient_boosting",
    "target": target,
    "target_type": "delta_converted_to_actual",
    "params": json.dumps(params),
    "train_rows": result["train_rows"],
    "evaluation_rows": len(validation_delta),
    "fit_seconds": result["fit_seconds"],
    **actual_metrics,
})

delta_validation_df = pd.DataFrame(delta_rows).sort_values(["target", "rmse"])
display(delta_validation_df)
delta_validation_df.to_csv(RESULT_DIR / "hgb_ura_delta_validation_tuning.csv", index=False)

,model,target,target_type,params,train_rows,evaluation_rows,fit_seconds,mae,rmse,r2
0,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.08, ""max_iter"": 350, ""max_...",2683410,574718,44.459255,1.10468,1.702315,0.851381


## 10. Focused HGB Delta Search

The first tuning pass found a strong HGB candidate and the delta target improved validation RMSE. This second pass searches a compact neighborhood around the winning HGB settings, keeping the same split and evaluating all candidates on the actual `target_next_URA` scale.

In [15]:
FOCUSED_HGB_DELTA_GRID = [
    {"learning_rate": 0.06, "max_iter": 500, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.08, "max_iter": 400, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.08, "max_iter": 500, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.10, "max_iter": 300, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.10, "max_iter": 400, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.12, "max_iter": 300, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 95, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 127, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.10, "max_iter": 350, "max_leaf_nodes": 95, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.10, "max_iter": 350, "max_leaf_nodes": 127, "min_samples_leaf": 100, "l2_regularization": 0.05},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 63, "min_samples_leaf": 75, "l2_regularization": 0.05},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 63, "min_samples_leaf": 150, "l2_regularization": 0.05},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 63, "min_samples_leaf": 200, "l2_regularization": 0.05},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.03},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.10},
    {"learning_rate": 0.08, "max_iter": 350, "max_leaf_nodes": 63, "min_samples_leaf": 100, "l2_regularization": 0.20},
    {"learning_rate": 0.10, "max_iter": 400, "max_leaf_nodes": 95, "min_samples_leaf": 150, "l2_regularization": 0.10},
    {"learning_rate": 0.10, "max_iter": 400, "max_leaf_nodes": 127, "min_samples_leaf": 150, "l2_regularization": 0.10},
    {"learning_rate": 0.06, "max_iter": 650, "max_leaf_nodes": 95, "min_samples_leaf": 150, "l2_regularization": 0.10},
    {"learning_rate": 0.12, "max_iter": 300, "max_leaf_nodes": 95, "min_samples_leaf": 150, "l2_regularization": 0.10},
]

focused_rows = []
seen_params = set()
for i, params in enumerate(FOCUSED_HGB_DELTA_GRID, start=1):
    params_key = json.dumps(params, sort_keys=True)
    if params_key in seen_params:
        continue
    seen_params.add(params_key)
    print(f"focused HGB delta {i}/{len(FOCUSED_HGB_DELTA_GRID)} {params}")
    _, result, pred_delta = fit_eval(make_hgb(params), delta_col, train_delta, validation_delta, HGB_TRAIN_LIMIT)
    pred_actual = validation_delta[current_col].to_numpy() + pred_delta
    actual_metrics = metrics(validation_delta[target], pred_actual)
    row = {
        "model": "hist_gradient_boosting",
        "target": target,
        "target_type": "delta_converted_to_actual",
        "candidate": i,
        "params": json.dumps(params),
        "train_rows": result["train_rows"],
        "evaluation_rows": len(validation_delta),
        "fit_seconds": result["fit_seconds"],
        **actual_metrics,
    }
    focused_rows.append(row)
    display(pd.DataFrame([row]))

focused_delta_df = pd.DataFrame(focused_rows).sort_values("rmse")
display(focused_delta_df)
focused_delta_df.to_csv(RESULT_DIR / "hgb_ura_delta_focused_tuning.csv", index=False)

all_delta_candidates = pd.concat([delta_validation_df, focused_delta_df], ignore_index=True, sort=False).sort_values("rmse")
display(all_delta_candidates.head(10))
all_delta_candidates.to_csv(RESULT_DIR / "hgb_ura_delta_all_tuning.csv", index=False)

focused HGB delta 1/20 {'learning_rate': 0.06, 'max_iter': 500, 'max_leaf_nodes': 63, 'min_samples_leaf': 100, 'l2_regularization': 0.05}


,model,target,target_type,candidate,params,train_rows,evaluation_rows,fit_seconds,mae,rmse,r2
0,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,1,"{""learning_rate"": 0.06, ""max_iter"": 500, ""max_...",2683410,574718,56.576462,1.102194,1.69981,0.851818


focused HGB delta 2/20 {'learning_rate': 0.08, 'max_iter': 400, 'max_leaf_nodes': 63, 'min_samples_leaf': 100, 'l2_regularization': 0.05}


,model,target,target_type,candidate,params,train_rows,evaluation_rows,fit_seconds,mae,rmse,r2
0,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,2,"{""learning_rate"": 0.08, ""max_iter"": 400, ""max_...",2683410,574718,45.627244,1.099003,1.695183,0.852623


focused HGB delta 3/20 {'learning_rate': 0.08, 'max_iter': 500, 'max_leaf_nodes': 63, 'min_samples_leaf': 100, 'l2_regularization': 0.05}


...

,model,target,target_type,candidate,params,train_rows,evaluation_rows,fit_seconds,mae,rmse,r2
0,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,19,"{""learning_rate"": 0.06, ""max_iter"": 650, ""max_...",2683410,574718,72.813696,1.069874,1.660783,0.858544


focused HGB delta 20/20 {'learning_rate': 0.12, 'max_iter': 300, 'max_leaf_nodes': 95, 'min_samples_leaf': 150, 'l2_regularization': 0.1}


,model,target,target_type,candidate,params,train_rows,evaluation_rows,fit_seconds,mae,rmse,r2
0,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,20,"{""learning_rate"": 0.12, ""max_iter"": 300, ""max_...",2683410,574718,39.134411,1.075782,1.668061,0.857301


,model,target,target_type,candidate,params,train_rows,evaluation_rows,fit_seconds,mae,rmse,r2
17,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,18,"{""learning_rate"": 0.1, ""max_iter"": 400, ""max_l...",2683410,574718,51.376046,1.056438,1.645404,0.861152
9,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,10,"{""learning_rate"": 0.1, ""max_iter"": 350, ""max_l...",2683410,574718,46.152275,1.062200,1.649290,0.860495
18,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,19,"{""learning_rate"": 0.06, ""max_iter"": 650, ""max_...",2683410,574718,72.813696,1.069874,1.660783,0.858544
16,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,17,"{""learning_rate"": 0.1, ""max_iter"": 400, ""max_l...",2683410,574718,49.772665,1.068760,1.661052,0.858498
7,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,8,"{""learning_rate"": 0.08, ""max_iter"": 350, ""max_...",2683410,574718,48.879364,1.073049,1.662577,0.858238
8,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,9,"{""learning_rate"": 0.1, ""max_iter"": 350, ""max_l...",2683410,574718,44.827987,1.075491,1.666525,0.857564
19,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,20,"{""learning_rate"": 0.12, ""max_iter"": 300, ""max_...",2683410,574718,39.134411,1.075782,1.668061,0.857301
6,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,7,"{""learning_rate"": 0.08, ""max_iter"": 350, ""max_...",2683410,574718,45.145880,1.086370,1.679845,0.855278
2,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,3,"{""learning_rate"": 0.08, ""max_iter"": 500, ""max_...",2683410,574718,53.646091,1.089145,1.683035,0.854728
4,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,5,"{""learning_rate"": 0.1, ""max_iter"": 400, ""max_l...",2683410,574718,44.840221,1.089515,1.684655,0.854448


,model,target,target_type,params,train_rows,evaluation_rows,fit_seconds,mae,rmse,r2,candidate
1,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.1, ""max_iter"": 400, ""max_l...",2683410,574718,51.376046,1.056438,1.645404,0.861152,18.0
2,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.1, ""max_iter"": 350, ""max_l...",2683410,574718,46.152275,1.062200,1.649290,0.860495,10.0
3,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.06, ""max_iter"": 650, ""max_...",2683410,574718,72.813696,1.069874,1.660783,0.858544,19.0
4,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.1, ""max_iter"": 400, ""max_l...",2683410,574718,49.772665,1.068760,1.661052,0.858498,17.0
5,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.08, ""max_iter"": 350, ""max_...",2683410,574718,48.879364,1.073049,1.662577,0.858238,8.0
6,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.1, ""max_iter"": 350, ""max_l...",2683410,574718,44.827987,1.075491,1.666525,0.857564,9.0
7,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.12, ""max_iter"": 300, ""max_...",2683410,574718,39.134411,1.075782,1.668061,0.857301,20.0
8,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.08, ""max_iter"": 350, ""max_...",2683410,574718,45.145880,1.086370,1.679845,0.855278,7.0
9,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.08, ""max_iter"": 500, ""max_...",2683410,574718,53.646091,1.089145,1.683035,0.854728,3.0
10,hist_gradient_boosting,target_next_URA,delta_converted_to_actual,"{""learning_rate"": 0.1, ""max_iter"": 400, ""max_l...",2683410,574718,44.840221,1.089515,1.684655,0.854448,5.0


## 11. Final Test Evaluation of URA-Tuned Models

In [16]:
test_rows = []
test_predictions = []

for _, row in best_validation.iterrows():
    model = row["model"]
    params = json.loads(row["params"])
    estimator = make_hgb(params) if model == "hist_gradient_boosting" else make_rf(params)

    if model == "hist_gradient_boosting":
        train_limit = HGB_TRAIN_LIMIT
    elif ALLOW_FULL_RF_FINAL_TRAINING:
        train_limit = RF_FINAL_TRAIN_LIMIT
    else:
        train_limit = RF_TUNING_TRAIN_LIMIT

    # Primary URA evaluation.
    target = TUNING_TARGET
    train_mode = "full_or_configured" if train_limit is None else f"sample_{train_limit}"
    print(f"Final URA test fit: {model} {params} | train_mode={train_mode}")
    pipe, result, pred = fit_eval(estimator, target, train_val_frame, test_frame, train_limit)
    test_rows.append({
        "model": model,
        "target": target,
        "target_role": "tuned_primary",
        "target_type": "direct",
        "train_mode": train_mode,
        "params": json.dumps(params),
        **result,
    })
    tmp = test_frame[["Segment_ID", "ELY", "event_date", "target_horizon_days", target, "URA", "IRI"]].copy()
    tmp["model"] = model
    tmp["target"] = target
    tmp["target_role"] = "tuned_primary"
    tmp["target_type"] = "direct"
    tmp["train_mode"] = train_mode
    tmp["prediction"] = pred
    test_predictions.append(tmp)

    # Optional reference IRI evaluation. Hyperparameters are not tuned on IRI.
    if EVALUATE_REFERENCE_TARGETS:
        for ref_target in REFERENCE_TARGETS:
            print(f"Reference test fit with URA-tuned params: {model} {ref_target} {params} | train_mode={train_mode}")
            ref_estimator = make_hgb(params) if model == "hist_gradient_boosting" else make_rf(params)
            ref_pipe, ref_result, ref_pred = fit_eval(ref_estimator, ref_target, train_val_frame, test_frame, train_limit)
            test_rows.append({
                "model": model,
                "target": ref_target,
                "target_role": "reference_not_tuned",
                "target_type": "direct",
                "train_mode": train_mode,
                "params": json.dumps(params),
                **ref_result,
            })
            ref_tmp = test_frame[["Segment_ID", "ELY", "event_date", "target_horizon_days", ref_target, "URA", "IRI"]].copy()
            ref_tmp["model"] = model
            ref_tmp["target"] = ref_target
            ref_tmp["target_role"] = "reference_not_tuned"
            ref_tmp["target_type"] = "direct"
            ref_tmp["train_mode"] = train_mode
            ref_tmp["prediction"] = ref_pred
            test_predictions.append(ref_tmp)

test_results_df = pd.DataFrame(test_rows).sort_values(["target", "target_role", "rmse"])
display(test_results_df)
test_results_df.to_csv(RESULT_DIR / "final_ura_tuned_test_results.csv", index=False)
pd.concat(test_predictions, ignore_index=True).to_parquet(RESULT_DIR / "final_ura_tuned_test_predictions.parquet", index=False)

Final URA test fit: hist_gradient_boosting {'learning_rate': 0.08, 'max_iter': 350, 'max_leaf_nodes': 63, 'min_samples_leaf': 100, 'l2_regularization': 0.05} | train_mode=full_or_configured
Reference test fit with URA-tuned params: hist_gradient_boosting target_next_IRI {'learning_rate': 0.08, 'max_iter': 350, 'max_leaf_nodes': 63, 'min_samples_leaf': 100, 'l2_regularization': 0.05} | train_mode=full_or_configured
Final URA test fit: random_forest {'n_estimators': 350, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'max_depth': None} | train_mode=sample_600000
Reference test fit with URA-tuned params: random_forest target_next_IRI {'n_estimators': 350, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'max_depth': None} | train_mode=sample_600000


,model,target,target_role,target_type,train_mode,params,mae,rmse,r2,train_rows,evaluation_rows,fit_seconds
1,hist_gradient_boosting,target_next_IRI,reference_not_tuned,direct,full_or_configured,"{""learning_rate"": 0.08, ""max_iter"": 350, ""max_...",0.228339,0.400398,0.857245,3258128,575195,49.368615
3,random_forest,target_next_IRI,reference_not_tuned,direct,sample_600000,"{""n_estimators"": 350, ""max_features"": ""sqrt"", ...",0.231960,0.407196,0.852357,600000,575195,67.996404
0,hist_gradient_boosting,target_next_URA,tuned_primary,direct,full_or_configured,"{""learning_rate"": 0.08, ""max_iter"": 350, ""max_...",1.112824,1.716192,0.849478,3258128,575195,55.218180
2,random_forest,target_next_URA,tuned_primary,direct,sample_600000,"{""n_estimators"": 350, ""max_features"": ""sqrt"", ...",1.107865,1.737325,0.845749,600000,575195,61.051619


## 12. Horizon and ELY Breakdown for Final Candidates

In [17]:
final_preds = pd.concat(test_predictions, ignore_index=True)
final_preds["horizon_window"] = "outside_windows"
for name, (lo, hi) in HORIZON_WINDOWS.items():
    final_preds.loc[final_preds["target_horizon_days"].between(lo, hi, inclusive="both"), "horizon_window"] = name

rows = []
for keys, group in final_preds.groupby(["model", "target", "target_role", "target_type"], sort=False):
    model, target, target_role, target_type = keys
    for group_type, col in [("overall", None), ("horizon_window", "horizon_window"), ("ELY", "ELY")]:
        groups = [("overall", group)] if col is None else group.groupby(col, sort=True)
        for value, sub in groups:
            if value == "outside_windows":
                continue
            rows.append({
                "model": model,
                "target": target,
                "target_role": target_role,
                "target_type": target_type,
                "group_type": group_type,
                "group": value,
                "evaluation_rows": len(sub),
                **metrics(sub[target], sub["prediction"]),
            })

breakdown_df = pd.DataFrame(rows).sort_values(["target", "target_role", "group_type", "group", "rmse"])
display(breakdown_df)
breakdown_df.to_csv(RESULT_DIR / "final_ura_tuned_test_breakdowns.csv", index=False)

,model,target,target_role,target_type,group_type,group,evaluation_rows,mae,rmse,r2
19,hist_gradient_boosting,target_next_IRI,reference_not_tuned,direct,ELY,Epo,61939,0.238559,0.412496,0.854669
47,random_forest,target_next_IRI,reference_not_tuned,direct,ELY,Epo,61939,0.242961,0.420983,0.848627
20,hist_gradient_boosting,target_next_IRI,reference_not_tuned,direct,ELY,Kas,31743,0.193983,0.342018,0.878414
48,random_forest,target_next_IRI,reference_not_tuned,direct,ELY,Kas,31743,0.196070,0.344369,0.876737
21,hist_gradient_boosting,target_next_IRI,reference_not_tuned,direct,ELY,Kes,33919,0.211689,0.379173,0.850944
49,random_forest,target_next_IRI,reference_not_tuned,direct,ELY,Kes,33919,0.214739,0.384469,0.846751
22,hist_gradient_boosting,target_next_IRI,reference_not_tuned,direct,ELY,Lap,54470,0.266356,0.437717,0.837040
50,random_forest,target_next_IRI,reference_not_tuned,direct,ELY,Lap,54470,0.271521,0.447508,0.829668
23,hist_gradient_boosting,target_next_IRI,reference_not_tuned,direct,ELY,Pir,45640,0.213138,0.382602,0.872076
51,random_forest,target_next_IRI,reference_not_tuned,direct,ELY,Pir,45640,0.215664,0.387634,0.868689


## 13. Feature Importance for Final Tree Models

In [18]:
# RF exposes impurity importance directly. HGB does not expose a stable built-in feature importance,
# so use permutation importance later if detailed HGB interpretation is needed.
print("For final reporting, consider permutation importance for the selected HGB model after tuning.")
print("RF feature importances can be extracted from fitted pipelines if needed in a later interpretation notebook.")

For final reporting, consider permutation importance for the selected HGB model after tuning.
RF feature importances can be extracted from fitted pipelines if needed in a later interpretation notebook.


## 14. Final Selection Summary

In [19]:
primary_rows = test_results_df[test_results_df["target_role"].eq("tuned_primary")].sort_values("rmse")
best_direct_test = primary_rows.iloc[0]
print(f"Best direct tuned-model test result in this notebook: {best_direct_test['model']} with RMSE={best_direct_test['rmse']:.4f}, MAE={best_direct_test['mae']:.4f}, R2={best_direct_test['r2']:.4f}")

validation_candidate_tables = [tuning_results.assign(target_type="direct"), delta_validation_df]
if "focused_delta_df" in globals():
    validation_candidate_tables.append(focused_delta_df)
final_validation_candidates = pd.concat(validation_candidate_tables, ignore_index=True, sort=False).sort_values("rmse")
print("\nBest validation candidate to carry into final training:")
display(final_validation_candidates.head(5))

if EVALUATE_REFERENCE_TARGETS:
    print("\nReference IRI results with URA-tuned hyperparameters:")
    display(test_results_df[test_results_df["target_role"].eq("reference_not_tuned")].sort_values("rmse"))

with open(RESULT_DIR / "ura_tuning_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "tuning_target": TUNING_TARGET,
        "reference_targets": REFERENCE_TARGETS,
        "evaluate_reference_targets": EVALUATE_REFERENCE_TARGETS,
        "features": FEATURES,
        "hgb_param_grid": HGB_PARAM_GRID,
        "focused_hgb_delta_grid": FOCUSED_HGB_DELTA_GRID,
        "rf_param_grid": RF_PARAM_GRID,
        "rf_tuning_train_limit": RF_TUNING_TRAIN_LIMIT,
        "rf_final_train_limit": RF_FINAL_TRAIN_LIMIT,
        "allow_full_rf_final_training": ALLOW_FULL_RF_FINAL_TRAINING,
        "hgb_train_limit": HGB_TRAIN_LIMIT,
        "random_state": RANDOM_STATE,
    }, f, indent=2)

print(f"Tuning artifacts saved to: {RESULT_DIR.resolve()}")

Best direct tuned-model test result in this notebook: hist_gradient_boosting with RMSE=1.7162, MAE=1.1128, R2=0.8495

Best validation candidate to carry into final training:


,model,target,candidate,params,mae,rmse,r2,train_rows,evaluation_rows,fit_seconds,target_type
10,hist_gradient_boosting,target_next_URA,18.0,"{""learning_rate"": 0.1, ""max_iter"": 400, ""max_l...",1.056438,1.645404,0.861152,2683410,574718,51.376046,delta_converted_to_actual
11,hist_gradient_boosting,target_next_URA,10.0,"{""learning_rate"": 0.1, ""max_iter"": 350, ""max_l...",1.062200,1.649290,0.860495,2683410,574718,46.152275,delta_converted_to_actual
12,hist_gradient_boosting,target_next_URA,19.0,"{""learning_rate"": 0.06, ""max_iter"": 650, ""max_...",1.069874,1.660783,0.858544,2683410,574718,72.813696,delta_converted_to_actual
13,hist_gradient_boosting,target_next_URA,17.0,"{""learning_rate"": 0.1, ""max_iter"": 400, ""max_l...",1.068760,1.661052,0.858498,2683410,574718,49.772665,delta_converted_to_actual
14,hist_gradient_boosting,target_next_URA,8.0,"{""learning_rate"": 0.08, ""max_iter"": 350, ""max_...",1.073049,1.662577,0.858238,2683410,574718,48.879364,delta_converted_to_actual



Reference IRI results with URA-tuned hyperparameters:


,model,target,target_role,target_type,train_mode,params,mae,rmse,r2,train_rows,evaluation_rows,fit_seconds
1,hist_gradient_boosting,target_next_IRI,reference_not_tuned,direct,full_or_configured,"{""learning_rate"": 0.08, ""max_iter"": 350, ""max_...",0.228339,0.400398,0.857245,3258128,575195,49.368615
3,random_forest,target_next_IRI,reference_not_tuned,direct,sample_600000,"{""n_estimators"": 350, ""max_features"": ""sqrt"", ...",0.231960,0.407196,0.852357,600000,575195,67.996404


Tuning artifacts saved to: C:\Users\Gamer2\Documents\school_sync\misc_maisteri_kurssit\case_studies_operations_research\Vayla_projekti\results\final_model_tuning
